In [ ]:
#@title Cell 32.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 32: Pathogen-Out Validation Across Models 3, 3B and C

## Purpose

Notebook 32 will compare Models 3, 3B and C using exactly the same pathogen-out
validation observations.

It will:

1. load the five BioSample-grouped outer folds defined in Notebook 26;
2. load the held-out Model 3B and Model C predictions from Notebook 26;
3. load the fixed Model 3 pathogen and antibiotic coordinates;
4. retain the same 9,058 Model C pathogens and 50,460 observed MIC values;
5. fit Model 3 on the training pathogens of each outer fold;
6. predict MIC only for the pathogens held out from that fold;
7. confirm that every observed MIC has one held-out prediction from each model;
8. calculate overall MAE, RMSE and Pearson correlation for all three models;
9. calculate performance separately for each outer fold and antibiotic;
10. calculate paired BioSample-bootstrap 95% intervals;
11. create a three-model pathogen-out comparison figure; and
12. validate and package the complete comparison outputs.

## Fair comparison

The same 9,058 pathogens, 50,460 observed MIC values and five held-out groups
will be used for all three models. Within each outer fold, the held-out
pathogens' MIC values are excluded from Model 3 fitting.

Notebook 26 already performed nested selection for Model 3B and Model C. Those
held-out predictions will be reused without modification. Model 3 has no
sequence-kernel choice: its established 32 pathogen coordinates and Ridge
penalty are read from the fixed Model 3 archive and applied to the same five
outer folds.

The BioSample bootstrap is a post-validation uncertainty calculation. It does
not refit or change any model.

## Expected notebook length

Notebook 32 contains **12 cells**.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 32 settings."
)


In [ ]:
#@title Cell 32.2 - Import packages and define notebook settings
# This cell imports the required packages, mounts Google Drive and defines the
# fixed cohort, fold, bootstrap and output settings.

from pathlib import Path
import hashlib
import json
import math
import shutil
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

drive.mount(
    "/content/drive",
    force_remount=False,
)

EXPECTED_MODEL_C_PATHOGENS = 9_058
EXPECTED_MODEL3_PATHOGENS = 9_377
EXPECTED_ANTIBIOTICS = 26
EXPECTED_INTERACTIONS = 50_460
EXPECTED_OUTER_FOLDS = 5
EXPECTED_NESTED_MODELS = 2
EXPECTED_COMPARISON_MODELS = 3

BOOTSTRAP_REPLICATES = 1_000
BOOTSTRAP_BATCH_SIZE = 50
RANDOM_SEED = 42

RIDGE_SOLVER = "lsqr"
RIDGE_TOLERANCE = 1e-4
RIDGE_MAXIMUM_ITERATIONS = 2_000

NOTEBOOK26_ARCHIVE_FILENAME = (
    "26_model_c_nested_pathogen_out_and_reference_model_outputs.zip"
)
MODEL3_ARCHIVE_FILENAME = (
    "16_model3_tensor_product_outputs.zip"
)

# Set an override only if an archive is outside Model3_MIC_Project or if
# different files with the same name are present.
NOTEBOOK26_ARCHIVE_PATH_OVERRIDE = ""
MODEL3_ARCHIVE_PATH_OVERRIDE = ""

MYDRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive"
)
PROJECT_DIRECTORY = (
    MYDRIVE_DIRECTORY
    / "Model3_MIC_Project"
)
NOTEBOOK32_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook32"
)
NOTEBOOK32_RESULT_DIRECTORY = (
    NOTEBOOK32_DIRECTORY
    / "results"
)
WORK_DIRECTORY = Path(
    "/content/notebook32_work"
)
INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "inputs"
)
NOTEBOOK26_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "notebook26"
)
MODEL3_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "model3"
)

for directory in [
    PROJECT_DIRECTORY,
    NOTEBOOK32_DIRECTORY,
    NOTEBOOK32_RESULT_DIRECTORY,
    WORK_DIRECTORY,
    INPUT_DIRECTORY,
    NOTEBOOK26_INPUT_DIRECTORY,
    MODEL3_INPUT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def file_sha256(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def safe_pearson_correlation(observed, predicted):
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)

    if len(observed) < 2:
        return np.nan
    if np.std(observed) == 0 or np.std(predicted) == 0:
        return np.nan

    return float(
        np.corrcoef(observed, predicted)[0, 1]
    )


def calculate_performance(observed, predicted):
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)

    return {
        "observations": len(observed),
        "mae": float(
            mean_absolute_error(observed, predicted)
        ),
        "rmse": float(math.sqrt(
            mean_squared_error(observed, predicted)
        )),
        "pearson_r": safe_pearson_correlation(
            observed,
            predicted,
        ),
    }


settings_summary = pd.DataFrame([
    {"setting": "Model C pathogens", "value": EXPECTED_MODEL_C_PATHOGENS},
    {"setting": "Observed MIC interactions", "value": EXPECTED_INTERACTIONS},
    {"setting": "Outer BioSample-grouped folds", "value": EXPECTED_OUTER_FOLDS},
    {"setting": "Models compared", "value": "Model 3, Model 3B and Model C"},
    {"setting": "BioSample bootstrap replicates", "value": BOOTSTRAP_REPLICATES},
    {"setting": "Notebook 32 output directory", "value": str(NOTEBOOK32_DIRECTORY)},
])

display(settings_summary)

print(
    "Notebook 32 packages, directories and settings were defined successfully."
)
print(
    "\nTransition: Cell 32.3 will locate, validate and extract "
    "the Notebook 26 and fixed Model 3 inputs."
)


In [ ]:
#@title Cell 32.3 - Locate, validate and extract the required archives
# This cell locates the Notebook 26 pathogen-out archive and the fixed Model 3
# archive, validates them and extracts only the files required here.


def locate_archive(authoritative_filename, override_path):
    if str(override_path).strip():
        archive_path = Path(str(override_path).strip())
        if not archive_path.exists():
            raise FileNotFoundError(
                f"The specified archive does not exist: {archive_path}"
            )
        return archive_path

    candidate_paths = sorted(
        {
            path.resolve()
            for path in PROJECT_DIRECTORY.rglob(authoritative_filename)
            if path.is_file()
        },
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    if not candidate_paths:
        raise FileNotFoundError(
            f"{authoritative_filename} was not found under "
            f"{PROJECT_DIRECTORY}. Copy it anywhere inside that "
            "project directory and rerun this cell."
        )

    if len(candidate_paths) > 1:
        candidate_hashes = {
            file_sha256(path)
            for path in candidate_paths
        }
        if len(candidate_hashes) > 1:
            display(pd.DataFrame({
                "candidate_path": [str(path) for path in candidate_paths],
                "sha256": [file_sha256(path) for path in candidate_paths],
            }))
            raise ValueError(
                f"Multiple different copies of {authoritative_filename} "
                "were found. Set its path override in Cell 32.2."
            )

    return candidate_paths[0]


def extract_required_members(
    archive_path,
    required_basenames,
    destination_directory,
):
    extracted_paths = {}

    with zipfile.ZipFile(archive_path, "r") as archive:
        damaged_member = archive.testzip()
        if damaged_member is not None:
            raise ValueError(
                f"{archive_path.name} contains a damaged member: "
                f"{damaged_member}"
            )

        basename_lookup = {}
        for member_name in archive.namelist():
            basename = Path(member_name).name
            if basename:
                basename_lookup.setdefault(
                    basename,
                    [],
                ).append(member_name)

        for required_basename in required_basenames:
            matches = basename_lookup.get(
                required_basename,
                [],
            )
            if len(matches) != 1:
                raise ValueError(
                    f"Expected one {required_basename} in "
                    f"{archive_path.name}; observed {len(matches)}."
                )

            output_path = (
                destination_directory
                / required_basename
            )
            partial_path = Path(
                str(output_path) + ".partial"
            )
            partial_path.unlink(missing_ok=True)

            with archive.open(matches[0], "r") as input_file:
                with open(partial_path, "wb") as output_file:
                    shutil.copyfileobj(
                        input_file,
                        output_file,
                        length=1024 * 1024,
                    )

            partial_path.replace(output_path)
            extracted_paths[required_basename] = output_path

    return extracted_paths


notebook26_archive_path = locate_archive(
    NOTEBOOK26_ARCHIVE_FILENAME,
    NOTEBOOK26_ARCHIVE_PATH_OVERRIDE,
)
model3_archive_path = locate_archive(
    MODEL3_ARCHIVE_FILENAME,
    MODEL3_ARCHIVE_PATH_OVERRIDE,
)

notebook26_files = extract_required_members(
    notebook26_archive_path,
    [
        "26_model_c_aligned_interactions.csv.gz",
        "26_biosample_outer_fold_assignments.csv",
        "26_nested_pathogen_out_predictions.csv.gz",
        "26_nested_outer_fold_results.csv",
        "26_nested_overall_performance.csv",
        "26_output_manifest.json",
    ],
    NOTEBOOK26_INPUT_DIRECTORY,
)

model3_files = extract_required_members(
    model3_archive_path,
    [
        "16_model3_kernel_embeddings.npz",
        "16_model3_antibiotic_embedding_index.csv",
        "16_model3_pathogen_embedding_index.csv",
        "16_model3_configuration.json",
    ],
    MODEL3_INPUT_DIRECTORY,
)

input_archive_summary = pd.DataFrame([
    {
        "input": "Notebook 26 pathogen-out results",
        "archive": notebook26_archive_path.name,
        "required_files": len(notebook26_files),
        "validation_status": "passed",
    },
    {
        "input": "Fixed Model 3 coordinates and settings",
        "archive": model3_archive_path.name,
        "required_files": len(model3_files),
        "validation_status": "passed",
    },
])

display(input_archive_summary)

print(f"Notebook 26 archive: {notebook26_archive_path}")
print(f"Model 3 archive: {model3_archive_path}")
print(
    "\nTransition: Cell 32.4 will load and validate the held-out "
    "predictions, fold assignments and fixed Model 3 coordinates."
)


In [ ]:
#@title Cell 32.4 - Load and validate the common cohort and model inputs
# This cell confirms that Notebook 26 contains complete held-out predictions
# for Model 3B and Model C and loads the fixed Model 3 coordinates and settings.

interactions = pd.read_csv(
    notebook26_files[
        "26_model_c_aligned_interactions.csv.gz"
    ]
).reset_index(drop=True)

biosample_fold_assignment = pd.read_csv(
    notebook26_files[
        "26_biosample_outer_fold_assignments.csv"
    ]
)

nested_predictions = pd.read_csv(
    notebook26_files[
        "26_nested_pathogen_out_predictions.csv.gz"
    ]
)

nested_outer_results = pd.read_csv(
    notebook26_files[
        "26_nested_outer_fold_results.csv"
    ]
)

notebook26_performance = pd.read_csv(
    notebook26_files[
        "26_nested_overall_performance.csv"
    ]
)

with open(
    notebook26_files["26_output_manifest.json"],
    "r",
    encoding="utf-8",
) as input_file:
    notebook26_manifest = json.load(input_file)

model3_pathogen_index = pd.read_csv(
    model3_files[
        "16_model3_pathogen_embedding_index.csv"
    ]
)

model3_antibiotic_index = pd.read_csv(
    model3_files[
        "16_model3_antibiotic_embedding_index.csv"
    ]
)

with open(
    model3_files["16_model3_configuration.json"],
    "r",
    encoding="utf-8",
) as input_file:
    model3_configuration = json.load(input_file)

with np.load(
    model3_files["16_model3_kernel_embeddings.npz"],
    allow_pickle=False,
) as embedding_archive:
    model3_full_pathogen_embedding = np.asarray(
        embedding_archive["pathogen_embedding"],
        dtype=np.float32,
    )
    model3_antibiotic_embedding = np.asarray(
        embedding_archive["antibiotic_embedding"],
        dtype=np.float32,
    )

MODEL3_PATHOGEN_COORDINATES = int(
    model3_configuration[
        "pathogen_embedding_dimensions"
    ]
)
MODEL3_ANTIBIOTIC_COORDINATES = int(
    model3_configuration[
        "antibiotic_embedding_dimensions"
    ]
)
MODEL3_RIDGE_ALPHA = float(
    model3_configuration[
        "ridge_alpha"
    ]
)
MODEL3_INTERACTION_PREDICTORS = int(
    MODEL3_PATHOGEN_COORDINATES
    * MODEL3_ANTIBIOTIC_COORDINATES
)

if notebook26_manifest.get("validation_status") != "passed":
    raise ValueError(
        "The Notebook 26 archive did not pass validation."
    )
if int(notebook26_manifest["model_c_pathogens"]) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError("Notebook 26 has the wrong pathogen count.")
if int(notebook26_manifest["observed_interactions"]) != EXPECTED_INTERACTIONS:
    raise ValueError("Notebook 26 has the wrong interaction count.")
if int(notebook26_manifest["outer_validation_folds"]) != EXPECTED_OUTER_FOLDS:
    raise ValueError("Notebook 26 has the wrong outer-fold count.")

required_interaction_columns = {
    "biosample",
    "antibiotic",
    "log2_mic",
    "model_c_pathogen_row",
    "antibiotic_coordinate_row",
}
if not required_interaction_columns.issubset(interactions.columns):
    raise ValueError(
        "The Notebook 26 interaction table is missing required columns."
    )

if len(interactions) != EXPECTED_INTERACTIONS:
    raise ValueError("The interaction table does not contain 50,460 rows.")
if interactions["biosample"].nunique() != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError("The interaction table does not contain 9,058 pathogens.")
if interactions["antibiotic"].nunique() != EXPECTED_ANTIBIOTICS:
    raise ValueError("The interaction table does not contain 26 antibiotics.")
if interactions.duplicated(["biosample", "antibiotic"]).any():
    raise ValueError("Duplicate BioSample-antibiotic observations were detected.")

interactions.insert(
    0,
    "interaction_row",
    np.arange(len(interactions), dtype=np.int64),
)

if len(biosample_fold_assignment) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError("The fold-assignment table does not contain 9,058 pathogens.")
if biosample_fold_assignment["biosample"].duplicated().any():
    raise ValueError("A BioSample was assigned to more than one outer fold.")
if set(biosample_fold_assignment["outer_fold"].astype(int)) != set(
    range(1, EXPECTED_OUTER_FOLDS + 1)
):
    raise ValueError("The fold-assignment table does not contain folds 1 to 5.")

required_prediction_columns = {
    "interaction_row",
    "biosample",
    "antibiotic",
    "observed_log2_mic",
    "predicted_log2_mic",
    "outer_fold",
    "model",
    "pathogen_coordinates",
    "alpha",
}
if not required_prediction_columns.issubset(nested_predictions.columns):
    raise ValueError(
        "The Notebook 26 held-out prediction table is missing required columns."
    )

expected_nested_rows = (
    EXPECTED_NESTED_MODELS
    * EXPECTED_INTERACTIONS
)
if len(nested_predictions) != expected_nested_rows:
    raise ValueError(
        f"Expected {expected_nested_rows:,} Notebook 26 held-out predictions; "
        f"observed {len(nested_predictions):,}."
    )

expected_nested_model_names = {
    "Model 3B baseline",
    "Model C",
}
if set(nested_predictions["model"]) != expected_nested_model_names:
    raise ValueError(
        "The Notebook 26 prediction table does not contain the expected models."
    )

if nested_predictions.duplicated(
    ["model", "interaction_row"]
).any():
    raise ValueError(
        "A Notebook 26 model has duplicate held-out predictions."
    )

for model_name in expected_nested_model_names:
    model_table = nested_predictions[
        nested_predictions["model"] == model_name
    ]
    if len(model_table) != EXPECTED_INTERACTIONS:
        raise ValueError(
            f"{model_name} does not contain 50,460 held-out predictions."
        )

prediction_observed_check = (
    nested_predictions[
        ["interaction_row", "observed_log2_mic"]
    ]
    .drop_duplicates()
    .sort_values("interaction_row")
    .reset_index(drop=True)
)

if not np.array_equal(
    prediction_observed_check["interaction_row"].to_numpy(np.int64),
    interactions["interaction_row"].to_numpy(np.int64),
):
    raise ValueError(
        "Notebook 26 predictions do not cover every aligned interaction."
    )

if not np.allclose(
    prediction_observed_check["observed_log2_mic"].to_numpy(np.float64),
    interactions["log2_mic"].to_numpy(np.float64),
):
    raise ValueError(
        "The observed MIC values differ between Notebook 26 input tables."
    )

if model3_full_pathogen_embedding.shape != (
    EXPECTED_MODEL3_PATHOGENS,
    MODEL3_PATHOGEN_COORDINATES,
):
    raise ValueError(
        "The Model 3 pathogen-coordinate matrix has the wrong dimensions."
    )
if model3_antibiotic_embedding.shape != (
    EXPECTED_ANTIBIOTICS,
    MODEL3_ANTIBIOTIC_COORDINATES,
):
    raise ValueError(
        "The Model 3 antibiotic-coordinate matrix has the wrong dimensions."
    )
if len(model3_pathogen_index) != EXPECTED_MODEL3_PATHOGENS:
    raise ValueError("The Model 3 pathogen index has the wrong row count.")
if len(model3_antibiotic_index) != EXPECTED_ANTIBIOTICS:
    raise ValueError("The Model 3 antibiotic index has the wrong row count.")

input_validation_summary = pd.DataFrame([
    {"metric": "Common pathogens", "value": EXPECTED_MODEL_C_PATHOGENS},
    {"metric": "Common observed MIC values", "value": EXPECTED_INTERACTIONS},
    {"metric": "Common outer folds", "value": EXPECTED_OUTER_FOLDS},
    {"metric": "Existing Model 3B predictions", "value": EXPECTED_INTERACTIONS},
    {"metric": "Existing Model C predictions", "value": EXPECTED_INTERACTIONS},
    {"metric": "Model 3 pathogen coordinates", "value": MODEL3_PATHOGEN_COORDINATES},
    {"metric": "Model 3 Ridge alpha", "value": MODEL3_RIDGE_ALPHA},
    {"metric": "Input validation status", "value": "passed"},
])

display(input_validation_summary)

print(
    "Notebook 26 predictions and fixed Model 3 inputs were validated."
)
print(
    "\nTransition: Cell 32.5 will align the Model 3 coordinates "
    "with the 9,058 pathogens and 26 antibiotics."
)


In [ ]:
#@title Cell 32.5 - Align Model 3 coordinates with the common cohort
# This cell aligns each Model C BioSample and antibiotic with its fixed Model 3
# coordinate row and confirms the Notebook 26 outer-fold assignments.

required_model3_pathogen_columns = {
    "kernel_row",
    "biosample",
}
required_model3_antibiotic_columns = {
    "kernel_row",
    "antibiotic",
}

if not required_model3_pathogen_columns.issubset(
    model3_pathogen_index.columns
):
    raise ValueError("The Model 3 pathogen index is missing required columns.")
if not required_model3_antibiotic_columns.issubset(
    model3_antibiotic_index.columns
):
    raise ValueError("The Model 3 antibiotic index is missing required columns.")

cohort_pathogens = (
    interactions[
        ["model_c_pathogen_row", "biosample"]
    ]
    .drop_duplicates()
    .sort_values("model_c_pathogen_row")
    .reset_index(drop=True)
)

if len(cohort_pathogens) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError("The common cohort does not contain 9,058 unique pathogens.")

cohort_pathogens = cohort_pathogens.merge(
    model3_pathogen_index[
        ["kernel_row", "biosample"]
    ].rename(columns={
        "kernel_row": "model3_pathogen_embedding_row",
    }),
    on="biosample",
    how="left",
    validate="one_to_one",
)

if cohort_pathogens["model3_pathogen_embedding_row"].isna().any():
    raise ValueError(
        "At least one Model C pathogen is absent from the Model 3 index."
    )

cohort_pathogens["model3_pathogen_embedding_row"] = (
    cohort_pathogens["model3_pathogen_embedding_row"].astype(int)
)

model3_cohort_pathogen_embedding = np.asarray(
    model3_full_pathogen_embedding[
        cohort_pathogens["model3_pathogen_embedding_row"].to_numpy(np.int64),
        :,
    ],
    dtype=np.float32,
)

if model3_cohort_pathogen_embedding.shape != (
    EXPECTED_MODEL_C_PATHOGENS,
    MODEL3_PATHOGEN_COORDINATES,
):
    raise ValueError(
        "The aligned Model 3 pathogen coordinates have the wrong dimensions."
    )

antibiotic_to_model3_row = dict(zip(
    model3_antibiotic_index["antibiotic"].astype(str),
    model3_antibiotic_index["kernel_row"].astype(int),
))

interactions["model3_antibiotic_row"] = (
    interactions["antibiotic"].astype(str).map(
        antibiotic_to_model3_row
    )
)

if interactions["model3_antibiotic_row"].isna().any():
    raise ValueError(
        "At least one Notebook 26 antibiotic is absent from the Model 3 index."
    )

interactions["model3_antibiotic_row"] = (
    interactions["model3_antibiotic_row"].astype(int)
)

fold_lookup = dict(zip(
    biosample_fold_assignment["biosample"].astype(str),
    biosample_fold_assignment["outer_fold"].astype(int),
))

interactions["outer_fold"] = (
    interactions["biosample"].astype(str).map(fold_lookup)
)

if interactions["outer_fold"].isna().any():
    raise ValueError(
        "At least one interaction could not be assigned to an outer fold."
    )

interactions["outer_fold"] = interactions["outer_fold"].astype(int)

nested_fold_check = (
    nested_predictions[
        ["interaction_row", "outer_fold"]
    ]
    .drop_duplicates()
    .sort_values("interaction_row")
    .reset_index(drop=True)
)

if len(nested_fold_check) != EXPECTED_INTERACTIONS:
    raise ValueError("Notebook 26 contains inconsistent outer-fold labels.")

if not np.array_equal(
    nested_fold_check["outer_fold"].to_numpy(np.int64),
    interactions["outer_fold"].to_numpy(np.int64),
):
    raise ValueError(
        "The reconstructed and saved Notebook 26 outer-fold labels differ."
    )

alignment_validation = pd.DataFrame([
    {"metric": "Model 3 reference pathogens", "value": EXPECTED_MODEL3_PATHOGENS},
    {"metric": "Aligned Model C pathogens", "value": len(cohort_pathogens)},
    {"metric": "Aligned antibiotics", "value": interactions["antibiotic"].nunique()},
    {"metric": "Aligned interactions", "value": len(interactions)},
    {"metric": "Outer-fold disagreements", "value": 0},
    {"metric": "Alignment validation status", "value": "passed"},
])

ALIGNMENT_VALIDATION_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_model3_alignment_validation.csv"
)
alignment_validation.to_csv(
    ALIGNMENT_VALIDATION_PATH,
    index=False,
)

display(alignment_validation)

print(f"Saved: {ALIGNMENT_VALIDATION_PATH}")
print(
    "\nTransition: Cell 32.6 will fit Model 3 on each outer "
    "training group and predict its held-out pathogens."
)


In [ ]:
#@title Cell 32.6 - Calculate Model 3 pathogen-out predictions
# This cell constructs the Model 3 interaction features once, fits Model 3 on
# each outer training group and predicts only the corresponding held-out group.

pathogen_coordinate_rows = interactions[
    "model_c_pathogen_row"
].to_numpy(np.int64)

antibiotic_coordinate_rows = interactions[
    "model3_antibiotic_row"
].to_numpy(np.int64)

interaction_pathogen_coordinates = (
    model3_cohort_pathogen_embedding[
        pathogen_coordinate_rows,
        :,
    ]
)

interaction_antibiotic_coordinates = (
    model3_antibiotic_embedding[
        antibiotic_coordinate_rows,
        :,
    ]
)

model3_design = np.einsum(
    "ir,is->irs",
    interaction_pathogen_coordinates,
    interaction_antibiotic_coordinates,
    optimize=True,
).reshape(
    EXPECTED_INTERACTIONS,
    MODEL3_INTERACTION_PREDICTORS,
).astype(
    np.float32,
    copy=False,
)

if model3_design.shape != (
    EXPECTED_INTERACTIONS,
    MODEL3_INTERACTION_PREDICTORS,
):
    raise ValueError("The Model 3 interaction matrix has the wrong dimensions.")
if not np.isfinite(model3_design).all():
    raise ValueError("The Model 3 interaction matrix contains invalid values.")

observed_values = interactions[
    "log2_mic"
].to_numpy(np.float64)

model3_prediction_tables = []
model3_fold_rows = []

for outer_fold in range(1, EXPECTED_OUTER_FOLDS + 1):
    evaluation_mask = (
        interactions["outer_fold"].to_numpy(np.int64)
        == outer_fold
    )
    training_mask = ~evaluation_mask

    training_biosamples = set(
        interactions.loc[training_mask, "biosample"].astype(str)
    )
    evaluation_biosamples = set(
        interactions.loc[evaluation_mask, "biosample"].astype(str)
    )

    if training_biosamples & evaluation_biosamples:
        raise ValueError(
            f"Outer fold {outer_fold} contains BioSample leakage."
        )

    training_indices = np.flatnonzero(training_mask)
    evaluation_indices = np.flatnonzero(evaluation_mask)

    training_design = np.asarray(
        model3_design[training_indices, :],
        dtype=np.float32,
    )
    evaluation_design = np.asarray(
        model3_design[evaluation_indices, :],
        dtype=np.float32,
    )

    model3_fold_model = Ridge(
        alpha=MODEL3_RIDGE_ALPHA,
        fit_intercept=True,
        solver=RIDGE_SOLVER,
        tol=RIDGE_TOLERANCE,
        max_iter=RIDGE_MAXIMUM_ITERATIONS,
        copy_X=False,
    )

    model3_fold_model.fit(
        training_design,
        observed_values[training_indices],
    )

    fold_predictions = model3_fold_model.predict(
        evaluation_design
    )

    if not np.isfinite(fold_predictions).all():
        raise ValueError(
            f"Model 3 produced invalid predictions in outer fold {outer_fold}."
        )

    fold_table = interactions.loc[
        evaluation_indices,
        [
            "interaction_row",
            "biosample",
            "antibiotic",
            "log2_mic",
            "model_c_pathogen_row",
            "antibiotic_coordinate_row",
            "outer_fold",
        ],
    ].copy()

    fold_table = fold_table.rename(columns={
        "log2_mic": "observed_log2_mic",
    })
    fold_table["model"] = "Model 3"
    fold_table["predicted_log2_mic"] = fold_predictions
    fold_table["candidate_id"] = "fixed_model3"
    fold_table["sequence_kernel"] = "not used"
    fold_table["rho"] = np.nan
    fold_table["pathogen_coordinates"] = MODEL3_PATHOGEN_COORDINATES
    fold_table["alpha"] = MODEL3_RIDGE_ALPHA

    model3_prediction_tables.append(fold_table)

    fold_performance = calculate_performance(
        observed_values[evaluation_indices],
        fold_predictions,
    )

    model3_fold_rows.append({
        "outer_fold": outer_fold,
        "training_biosamples": len(training_biosamples),
        "evaluation_biosamples": len(evaluation_biosamples),
        "training_interactions": len(training_indices),
        "evaluation_interactions": len(evaluation_indices),
        "biosample_overlap": 0,
        **fold_performance,
    })

    print(
        f"Outer fold {outer_fold}: "
        f"{len(evaluation_biosamples):,} held-out pathogens, "
        f"RMSE={fold_performance['rmse']:.4f}"
    )

    del training_design
    del evaluation_design
    del model3_fold_model

model3_pathogen_out_predictions = pd.concat(
    model3_prediction_tables,
    ignore_index=True,
).sort_values("interaction_row").reset_index(drop=True)

model3_outer_fold_performance = pd.DataFrame(
    model3_fold_rows
)

if len(model3_pathogen_out_predictions) != EXPECTED_INTERACTIONS:
    raise ValueError("Model 3 does not contain 50,460 held-out predictions.")
if model3_pathogen_out_predictions["interaction_row"].duplicated().any():
    raise ValueError("Model 3 has duplicate held-out predictions.")

MODEL3_PREDICTION_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_model3_pathogen_out_predictions.csv.gz"
)
MODEL3_FOLD_CHECK_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_model3_outer_fold_fit_check.csv"
)

model3_pathogen_out_predictions.to_csv(
    MODEL3_PREDICTION_PATH,
    index=False,
    compression="gzip",
)
model3_outer_fold_performance.to_csv(
    MODEL3_FOLD_CHECK_PATH,
    index=False,
)

display(model3_outer_fold_performance)

print(f"Saved: {MODEL3_PREDICTION_PATH}")
print(f"Saved: {MODEL3_FOLD_CHECK_PATH}")
print(
    "\nTransition: Cell 32.7 will combine the held-out "
    "predictions from all three models."
)


In [ ]:
#@title Cell 32.7 - Combine the three held-out prediction tables
# This cell combines Model 3 with the existing Model 3B and Model C predictions
# and confirms one held-out prediction per model and observed MIC value.

existing_predictions = nested_predictions.copy()
existing_predictions["model"] = existing_predictions["model"].replace({
    "Model 3B baseline": "Model 3B",
})

common_prediction_columns = [
    "interaction_row",
    "biosample",
    "antibiotic",
    "observed_log2_mic",
    "model_c_pathogen_row",
    "antibiotic_coordinate_row",
    "model",
    "predicted_log2_mic",
    "outer_fold",
    "candidate_id",
    "sequence_kernel",
    "rho",
    "pathogen_coordinates",
    "alpha",
]

for column_name in common_prediction_columns:
    if column_name not in existing_predictions.columns:
        raise ValueError(
            f"Notebook 26 predictions are missing {column_name}."
        )
    if column_name not in model3_pathogen_out_predictions.columns:
        raise ValueError(
            f"Model 3 predictions are missing {column_name}."
        )

three_model_predictions = pd.concat(
    [
        model3_pathogen_out_predictions[
            common_prediction_columns
        ],
        existing_predictions[
            common_prediction_columns
        ],
    ],
    ignore_index=True,
)

model_order = {
    "Model 3": 0,
    "Model 3B": 1,
    "Model C": 2,
}

three_model_predictions["model_order"] = (
    three_model_predictions["model"].map(model_order)
)

if three_model_predictions["model_order"].isna().any():
    raise ValueError("An unexpected model name was found.")

three_model_predictions = (
    three_model_predictions
    .sort_values(["interaction_row", "model_order"])
    .drop(columns="model_order")
    .reset_index(drop=True)
)

expected_combined_rows = (
    EXPECTED_COMPARISON_MODELS
    * EXPECTED_INTERACTIONS
)

if len(three_model_predictions) != expected_combined_rows:
    raise ValueError(
        f"Expected {expected_combined_rows:,} combined prediction rows; "
        f"observed {len(three_model_predictions):,}."
    )

if three_model_predictions.duplicated(
    ["model", "interaction_row"]
).any():
    raise ValueError(
        "At least one model has duplicate held-out predictions."
    )

prediction_counts = (
    three_model_predictions
    .groupby("interaction_row")["model"]
    .nunique()
)

if not (prediction_counts == EXPECTED_COMPARISON_MODELS).all():
    raise ValueError(
        "At least one observed MIC lacks a prediction from one model."
    )

observed_counts = (
    three_model_predictions
    .groupby("interaction_row")["observed_log2_mic"]
    .nunique()
)

if not (observed_counts == 1).all():
    raise ValueError(
        "The models do not share identical observed MIC values."
    )

fold_counts = (
    three_model_predictions
    .groupby("interaction_row")["outer_fold"]
    .nunique()
)

if not (fold_counts == 1).all():
    raise ValueError(
        "The models do not share identical outer-fold assignments."
    )

THREE_MODEL_PREDICTION_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_pathogen_out_predictions.csv.gz"
)

three_model_predictions.to_csv(
    THREE_MODEL_PREDICTION_PATH,
    index=False,
    compression="gzip",
)

combined_prediction_summary = pd.DataFrame([
    {
        "model": model_name,
        "held_out_predictions": len(model_table),
        "pathogens": model_table["biosample"].nunique(),
        "outer_folds": model_table["outer_fold"].nunique(),
    }
    for model_name, model_table in three_model_predictions.groupby(
        "model",
        sort=False,
    )
])

display(combined_prediction_summary)

print(f"Saved: {THREE_MODEL_PREDICTION_PATH}")
print(
    "\nTransition: Cell 32.8 will calculate overall, outer-fold "
    "and antibiotic-specific performance for all three models."
)


In [ ]:
#@title Cell 32.8 - Calculate three-model pathogen-out performance
# This cell calculates overall, outer-fold and antibiotic-specific MAE, RMSE
# and Pearson correlation and summarises variation across the five folds.

overall_performance_rows = []
outer_fold_performance_rows = []
antibiotic_performance_rows = []

for model_name, model_table in three_model_predictions.groupby(
    "model",
    sort=False,
):
    overall_metrics = calculate_performance(
        model_table["observed_log2_mic"],
        model_table["predicted_log2_mic"],
    )

    overall_performance_rows.append({
        "model": model_name,
        "pathogens": model_table["biosample"].nunique(),
        **overall_metrics,
    })

    for outer_fold, fold_table in model_table.groupby(
        "outer_fold",
        sort=True,
    ):
        fold_metrics = calculate_performance(
            fold_table["observed_log2_mic"],
            fold_table["predicted_log2_mic"],
        )

        outer_fold_performance_rows.append({
            "model": model_name,
            "outer_fold": int(outer_fold),
            "pathogens": fold_table["biosample"].nunique(),
            **fold_metrics,
        })

    for antibiotic_name, antibiotic_table in model_table.groupby(
        "antibiotic",
        sort=True,
    ):
        antibiotic_metrics = calculate_performance(
            antibiotic_table["observed_log2_mic"],
            antibiotic_table["predicted_log2_mic"],
        )

        antibiotic_performance_rows.append({
            "model": model_name,
            "antibiotic": antibiotic_name,
            "pathogens": antibiotic_table["biosample"].nunique(),
            **antibiotic_metrics,
        })

overall_performance = pd.DataFrame(
    overall_performance_rows
)
outer_fold_performance = pd.DataFrame(
    outer_fold_performance_rows
)
antibiotic_performance = pd.DataFrame(
    antibiotic_performance_rows
)

overall_performance["model_order"] = (
    overall_performance["model"].map(model_order)
)
overall_performance = (
    overall_performance
    .sort_values("model_order")
    .drop(columns="model_order")
    .reset_index(drop=True)
)

outer_fold_performance["model_order"] = (
    outer_fold_performance["model"].map(model_order)
)
outer_fold_performance = (
    outer_fold_performance
    .sort_values(["model_order", "outer_fold"])
    .drop(columns="model_order")
    .reset_index(drop=True)
)

fold_variation_rows = []
for model_name, model_table in outer_fold_performance.groupby(
    "model",
    sort=False,
):
    for metric_name in ["mae", "rmse", "pearson_r"]:
        metric_values = model_table[metric_name].to_numpy(np.float64)
        fold_variation_rows.append({
            "model": model_name,
            "metric": metric_name,
            "outer_folds": len(metric_values),
            "mean_across_folds": float(np.mean(metric_values)),
            "standard_deviation_across_folds": float(
                np.std(metric_values, ddof=1)
            ),
            "minimum_fold_value": float(np.min(metric_values)),
            "maximum_fold_value": float(np.max(metric_values)),
        })

outer_fold_variation = pd.DataFrame(
    fold_variation_rows
)

performance_lookup = overall_performance.set_index("model")

pairwise_definitions = [
    ("Model 3", "Model 3B"),
    ("Model 3B", "Model C"),
    ("Model 3", "Model C"),
]

pairwise_rows = []
for first_model, second_model in pairwise_definitions:
    first_metrics = performance_lookup.loc[first_model]
    second_metrics = performance_lookup.loc[second_model]

    pairwise_rows.append({
        "first_model": first_model,
        "second_model": second_model,
        "positive_value_favours": second_model,
        "mae_improvement": float(
            first_metrics["mae"]
            - second_metrics["mae"]
        ),
        "rmse_improvement": float(
            first_metrics["rmse"]
            - second_metrics["rmse"]
        ),
        "pearson_r_improvement": float(
            second_metrics["pearson_r"]
            - first_metrics["pearson_r"]
        ),
    })

pairwise_comparison = pd.DataFrame(pairwise_rows)

OVERALL_PERFORMANCE_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_overall_performance.csv"
)
OUTER_FOLD_PERFORMANCE_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_outer_fold_performance.csv"
)
OUTER_FOLD_VARIATION_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_outer_fold_variation.csv"
)
ANTIBIOTIC_PERFORMANCE_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_antibiotic_performance.csv"
)
PAIRWISE_COMPARISON_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_pairwise_comparison.csv"
)

overall_performance.to_csv(
    OVERALL_PERFORMANCE_PATH,
    index=False,
)
outer_fold_performance.to_csv(
    OUTER_FOLD_PERFORMANCE_PATH,
    index=False,
)
outer_fold_variation.to_csv(
    OUTER_FOLD_VARIATION_PATH,
    index=False,
)
antibiotic_performance.to_csv(
    ANTIBIOTIC_PERFORMANCE_PATH,
    index=False,
)
pairwise_comparison.to_csv(
    PAIRWISE_COMPARISON_PATH,
    index=False,
)

display(overall_performance.round(4))
display(pairwise_comparison.round(4))
display(outer_fold_variation.round(4))

print(
    "\nTransition: Cell 32.9 will calculate paired "
    "BioSample-bootstrap 95% intervals."
)


In [ ]:
#@title Cell 32.9 - Calculate paired BioSample-bootstrap intervals
# This cell resamples complete BioSamples, preserving all of each pathogen's
# MIC observations and the pairing among the three held-out prediction sets.

prediction_wide = (
    three_model_predictions
    .pivot(
        index=[
            "interaction_row",
            "biosample",
            "antibiotic",
            "observed_log2_mic",
            "outer_fold",
        ],
        columns="model",
        values="predicted_log2_mic",
    )
    .reset_index()
)

for model_name in model_order:
    if model_name not in prediction_wide.columns:
        raise ValueError(
            f"The paired prediction table is missing {model_name}."
        )

bootstrap_working = pd.DataFrame({
    "biosample": prediction_wide["biosample"].astype(str),
    "count": np.ones(len(prediction_wide), dtype=np.int64),
    "sum_observed": prediction_wide["observed_log2_mic"].to_numpy(np.float64),
    "sum_observed_squared": np.square(
        prediction_wide["observed_log2_mic"].to_numpy(np.float64)
    ),
})

bootstrap_model_keys = {
    "Model 3": "model3",
    "Model 3B": "model3b",
    "Model C": "modelc",
}

observed_bootstrap_values = prediction_wide[
    "observed_log2_mic"
].to_numpy(np.float64)

for model_name, model_key in bootstrap_model_keys.items():
    predicted_values = prediction_wide[model_name].to_numpy(np.float64)
    residual_values = observed_bootstrap_values - predicted_values

    bootstrap_working[f"{model_key}_sum_absolute_error"] = np.abs(
        residual_values
    )
    bootstrap_working[f"{model_key}_sum_squared_error"] = np.square(
        residual_values
    )
    bootstrap_working[f"{model_key}_sum_predicted"] = predicted_values
    bootstrap_working[f"{model_key}_sum_predicted_squared"] = np.square(
        predicted_values
    )
    bootstrap_working[f"{model_key}_sum_observed_predicted"] = (
        observed_bootstrap_values * predicted_values
    )

pathogen_metric_sums = (
    bootstrap_working
    .groupby("biosample", sort=True)
    .sum(numeric_only=True)
)

if len(pathogen_metric_sums) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        "The bootstrap summary does not contain 9,058 pathogens."
    )

rng = np.random.default_rng(RANDOM_SEED)
number_of_pathogens = len(pathogen_metric_sums)
bootstrap_tables = []

count_values = pathogen_metric_sums["count"].to_numpy(np.float64)
sum_observed_values = pathogen_metric_sums[
    "sum_observed"
].to_numpy(np.float64)
sum_observed_squared_values = pathogen_metric_sums[
    "sum_observed_squared"
].to_numpy(np.float64)

for batch_start in range(
    0,
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_BATCH_SIZE,
):
    batch_stop = min(
        batch_start + BOOTSTRAP_BATCH_SIZE,
        BOOTSTRAP_REPLICATES,
    )
    batch_size = batch_stop - batch_start

    sampled_pathogen_rows = rng.integers(
        0,
        number_of_pathogens,
        size=(batch_size, number_of_pathogens),
    )

    sampled_count = count_values[
        sampled_pathogen_rows
    ].sum(axis=1)
    sampled_observed_sum = sum_observed_values[
        sampled_pathogen_rows
    ].sum(axis=1)
    sampled_observed_squared_sum = sum_observed_squared_values[
        sampled_pathogen_rows
    ].sum(axis=1)

    replicate_numbers = np.arange(
        batch_start + 1,
        batch_stop + 1,
        dtype=np.int64,
    )

    for model_name, model_key in bootstrap_model_keys.items():
        absolute_error_sum = pathogen_metric_sums[
            f"{model_key}_sum_absolute_error"
        ].to_numpy(np.float64)[sampled_pathogen_rows].sum(axis=1)

        squared_error_sum = pathogen_metric_sums[
            f"{model_key}_sum_squared_error"
        ].to_numpy(np.float64)[sampled_pathogen_rows].sum(axis=1)

        predicted_sum = pathogen_metric_sums[
            f"{model_key}_sum_predicted"
        ].to_numpy(np.float64)[sampled_pathogen_rows].sum(axis=1)

        predicted_squared_sum = pathogen_metric_sums[
            f"{model_key}_sum_predicted_squared"
        ].to_numpy(np.float64)[sampled_pathogen_rows].sum(axis=1)

        observed_predicted_sum = pathogen_metric_sums[
            f"{model_key}_sum_observed_predicted"
        ].to_numpy(np.float64)[sampled_pathogen_rows].sum(axis=1)

        mae_values = absolute_error_sum / sampled_count
        rmse_values = np.sqrt(squared_error_sum / sampled_count)

        correlation_numerator = (
            sampled_count * observed_predicted_sum
            - sampled_observed_sum * predicted_sum
        )
        correlation_denominator = np.sqrt(
            (
                sampled_count * sampled_observed_squared_sum
                - np.square(sampled_observed_sum)
            )
            * (
                sampled_count * predicted_squared_sum
                - np.square(predicted_sum)
            )
        )
        pearson_values = np.divide(
            correlation_numerator,
            correlation_denominator,
            out=np.full(batch_size, np.nan, dtype=np.float64),
            where=correlation_denominator > 0,
        )

        bootstrap_tables.append(pd.DataFrame({
            "bootstrap_replicate": replicate_numbers,
            "model": model_name,
            "mae": mae_values,
            "rmse": rmse_values,
            "pearson_r": pearson_values,
        }))

bootstrap_performance = pd.concat(
    bootstrap_tables,
    ignore_index=True,
)

if not np.isfinite(
    bootstrap_performance[
        ["mae", "rmse", "pearson_r"]
    ].to_numpy(np.float64)
).all():
    raise ValueError(
        "The bootstrap performance table contains invalid values."
    )

bootstrap_interval_rows = []
for model_name in model_order:
    model_bootstrap = bootstrap_performance[
        bootstrap_performance["model"] == model_name
    ]
    point_metrics = performance_lookup.loc[model_name]

    for metric_name in ["mae", "rmse", "pearson_r"]:
        bootstrap_values = model_bootstrap[
            metric_name
        ].to_numpy(np.float64)

        bootstrap_interval_rows.append({
            "model": model_name,
            "metric": metric_name,
            "point_estimate": float(point_metrics[metric_name]),
            "lower_95_percentile": float(
                np.percentile(bootstrap_values, 2.5)
            ),
            "upper_95_percentile": float(
                np.percentile(bootstrap_values, 97.5)
            ),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
            "resampling_unit": "BioSample",
        })

bootstrap_intervals = pd.DataFrame(
    bootstrap_interval_rows
)

bootstrap_wide = bootstrap_performance.pivot(
    index="bootstrap_replicate",
    columns="model",
    values=["mae", "rmse", "pearson_r"],
)

pairwise_bootstrap_rows = []
for first_model, second_model in pairwise_definitions:
    for metric_name in ["mae", "rmse", "pearson_r"]:
        if metric_name in ["mae", "rmse"]:
            bootstrap_improvement = (
                bootstrap_wide[(metric_name, first_model)]
                - bootstrap_wide[(metric_name, second_model)]
            ).to_numpy(np.float64)
            point_improvement = float(
                performance_lookup.loc[first_model, metric_name]
                - performance_lookup.loc[second_model, metric_name]
            )
        else:
            bootstrap_improvement = (
                bootstrap_wide[(metric_name, second_model)]
                - bootstrap_wide[(metric_name, first_model)]
            ).to_numpy(np.float64)
            point_improvement = float(
                performance_lookup.loc[second_model, metric_name]
                - performance_lookup.loc[first_model, metric_name]
            )

        pairwise_bootstrap_rows.append({
            "first_model": first_model,
            "second_model": second_model,
            "metric": metric_name,
            "positive_value_favours": second_model,
            "point_improvement": point_improvement,
            "lower_95_percentile": float(
                np.percentile(bootstrap_improvement, 2.5)
            ),
            "upper_95_percentile": float(
                np.percentile(bootstrap_improvement, 97.5)
            ),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
            "resampling_unit": "BioSample",
        })

pairwise_bootstrap_intervals = pd.DataFrame(
    pairwise_bootstrap_rows
)

BOOTSTRAP_PERFORMANCE_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_biosample_bootstrap_performance.csv.gz"
)
BOOTSTRAP_INTERVAL_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_biosample_bootstrap_intervals.csv"
)
PAIRWISE_BOOTSTRAP_INTERVAL_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_pairwise_bootstrap_intervals.csv"
)

bootstrap_performance.to_csv(
    BOOTSTRAP_PERFORMANCE_PATH,
    index=False,
    compression="gzip",
)
bootstrap_intervals.to_csv(
    BOOTSTRAP_INTERVAL_PATH,
    index=False,
)
pairwise_bootstrap_intervals.to_csv(
    PAIRWISE_BOOTSTRAP_INTERVAL_PATH,
    index=False,
)

display(bootstrap_intervals.round(4))
display(pairwise_bootstrap_intervals.round(4))

print(
    "\nTransition: Cell 32.10 will create the three-model "
    "pathogen-out validation figure."
)


In [ ]:
#@title Cell 32.10 - Create the three-model pathogen-out validation figure
# This cell plots the overall MAE and RMSE with BioSample-bootstrap 95%
# intervals and shows RMSE variation across the five held-out groups.

model_display_order = [
    "Model 3",
    "Model 3B",
    "Model C",
]
model_colours = {
    "Model 3": "#4C78A8",
    "Model 3B": "#F58518",
    "Model C": "#54A24B",
}

figure, (overall_axis, fold_axis) = plt.subplots(
    1,
    2,
    figsize=(13, 5.5),
)

x_positions = np.arange(len(model_display_order))
bar_width = 0.36

for metric_name, offset, hatch in [
    ("mae", -bar_width / 2, ""),
    ("rmse", bar_width / 2, "//"),
]:
    metric_rows = (
        bootstrap_intervals[
            bootstrap_intervals["metric"] == metric_name
        ]
        .set_index("model")
        .loc[model_display_order]
    )

    point_values = metric_rows["point_estimate"].to_numpy(np.float64)
    lower_errors = (
        point_values
        - metric_rows["lower_95_percentile"].to_numpy(np.float64)
    )
    upper_errors = (
        metric_rows["upper_95_percentile"].to_numpy(np.float64)
        - point_values
    )

    overall_axis.bar(
        x_positions + offset,
        point_values,
        width=bar_width,
        label=metric_name.upper(),
        color=[model_colours[name] for name in model_display_order],
        edgecolor="black",
        linewidth=0.7,
        hatch=hatch,
        alpha=0.88,
        yerr=np.vstack([lower_errors, upper_errors]),
        capsize=4,
    )

overall_axis.set_xticks(x_positions)
overall_axis.set_xticklabels(model_display_order)
overall_axis.set_ylabel("Held-out prediction error in log2(MIC)")
overall_axis.set_title("Overall pathogen-out performance")
overall_axis.grid(axis="y", color="0.90", linewidth=0.8)
overall_axis.legend()

for model_name in model_display_order:
    model_fold_table = outer_fold_performance[
        outer_fold_performance["model"] == model_name
    ].sort_values("outer_fold")

    fold_axis.plot(
        model_fold_table["outer_fold"],
        model_fold_table["rmse"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model_name,
        color=model_colours[model_name],
    )

fold_axis.set_xticks(range(1, EXPECTED_OUTER_FOLDS + 1))
fold_axis.set_xlabel("Held-out outer fold")
fold_axis.set_ylabel("RMSE in log2(MIC)")
fold_axis.set_title("Variation across held-out pathogen groups")
fold_axis.grid(color="0.90", linewidth=0.8)
fold_axis.legend()

figure.suptitle(
    "Models 3, 3B and C on the same pathogen-out validation folds",
    fontsize=14,
)
figure.tight_layout(rect=[0, 0, 1, 0.95])

VALIDATION_FIGURE_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_pathogen_out_validation.png"
)

figure.savefig(
    VALIDATION_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

print(f"Saved: {VALIDATION_FIGURE_PATH}")
print(
    "\nTransition: Cell 32.11 will validate the complete "
    "three-model pathogen-out comparison."
)


In [ ]:
#@title Cell 32.11 - Validate the complete three-model comparison
# This cell verifies cohort identity, prediction completeness, numerical
# values, Notebook 26 reproduction and the final Notebook 32 configuration.

recalculated_lookup = overall_performance.set_index("model")
notebook26_lookup = notebook26_performance.set_index("model")

notebook26_model_name_map = {
    "Model 3B": "Model 3B baseline",
    "Model C": "Model C",
}

notebook26_metric_differences = []
for current_name, notebook26_name in notebook26_model_name_map.items():
    for metric_name in ["mae", "rmse", "pearson_r"]:
        notebook26_metric_differences.append(abs(
            float(recalculated_lookup.loc[current_name, metric_name])
            - float(notebook26_lookup.loc[notebook26_name, metric_name])
        ))

maximum_notebook26_metric_difference = float(
    max(notebook26_metric_differences)
)

numeric_prediction_values = three_model_predictions[
    ["observed_log2_mic", "predicted_log2_mic"]
].to_numpy(np.float64)

validation_checks = {
    "common_pathogen_count": (
        three_model_predictions["biosample"].nunique()
        == EXPECTED_MODEL_C_PATHOGENS
    ),
    "common_antibiotic_count": (
        three_model_predictions["antibiotic"].nunique()
        == EXPECTED_ANTIBIOTICS
    ),
    "common_interaction_count": (
        three_model_predictions["interaction_row"].nunique()
        == EXPECTED_INTERACTIONS
    ),
    "three_models_present": (
        set(three_model_predictions["model"])
        == set(model_display_order)
    ),
    "one_prediction_per_model_and_interaction": (
        not three_model_predictions.duplicated(
            ["model", "interaction_row"]
        ).any()
    ),
    "five_outer_folds_present": (
        set(three_model_predictions["outer_fold"].astype(int))
        == set(range(1, EXPECTED_OUTER_FOLDS + 1))
    ),
    "all_prediction_values_finite": (
        np.isfinite(numeric_prediction_values).all()
    ),
    "model3_no_biosample_leakage": (
        int(model3_outer_fold_performance["biosample_overlap"].max())
        == 0
    ),
    "notebook26_metrics_reproduced": (
        maximum_notebook26_metric_difference <= 1e-12
    ),
    "bootstrap_values_finite": (
        np.isfinite(
            bootstrap_performance[
                ["mae", "rmse", "pearson_r"]
            ].to_numpy(np.float64)
        ).all()
    ),
}

failed_checks = [
    check_name
    for check_name, passed in validation_checks.items()
    if not bool(passed)
]

if failed_checks:
    raise ValueError(
        f"Notebook 32 validation failed: {failed_checks}"
    )

VALIDATION_SUMMARY_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_validation_summary.csv"
)
CONFIGURATION_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_three_model_pathogen_out_configuration.json"
)

validation_summary = pd.DataFrame([
    {"metric": "Common pathogens", "value": EXPECTED_MODEL_C_PATHOGENS},
    {"metric": "Common observed MIC values", "value": EXPECTED_INTERACTIONS},
    {"metric": "Common antibiotics", "value": EXPECTED_ANTIBIOTICS},
    {"metric": "Common outer folds", "value": EXPECTED_OUTER_FOLDS},
    {"metric": "Held-out predictions", "value": len(three_model_predictions)},
    {"metric": "Model 3 pathogen coordinates", "value": MODEL3_PATHOGEN_COORDINATES},
    {"metric": "Model 3 Ridge alpha", "value": MODEL3_RIDGE_ALPHA},
    {"metric": "Maximum Notebook 26 metric difference", "value": maximum_notebook26_metric_difference},
    {"metric": "BioSample bootstrap replicates", "value": BOOTSTRAP_REPLICATES},
    {"metric": "Notebook 32 validation status", "value": "passed"},
])

validation_summary.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
)

configuration = {
    "notebook": 32,
    "purpose": "Pathogen-out validation across Models 3, 3B and C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "observed_interactions": EXPECTED_INTERACTIONS,
    "antibiotics": EXPECTED_ANTIBIOTICS,
    "outer_folds": EXPECTED_OUTER_FOLDS,
    "fold_source": "Notebook 26 BioSample-grouped outer folds",
    "model3": {
        "pathogen_coordinates": MODEL3_PATHOGEN_COORDINATES,
        "antibiotic_coordinates": MODEL3_ANTIBIOTIC_COORDINATES,
        "interaction_predictors": MODEL3_INTERACTION_PREDICTORS,
        "ridge_alpha": MODEL3_RIDGE_ALPHA,
        "selection": "fixed established Model 3 setting",
    },
    "model3b": {
        "prediction_source": "Notebook 26 nested pathogen-out baseline",
    },
    "model_c": {
        "prediction_source": "Notebook 26 nested pathogen-out Model C",
        "selected_sequence_kernel": notebook26_manifest[
            "selected_sequence_kernel"
        ],
        "selected_rho": float(
            notebook26_manifest["selected_rho"]
        ),
    },
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "resampling_unit": "BioSample",
        "random_seed": RANDOM_SEED,
        "interpretation": (
            "Post-validation uncertainty calculation; models were not refitted"
        ),
    },
    "validation_status": "passed",
}

temporary_configuration_path = Path(
    str(CONFIGURATION_PATH) + ".partial"
)
with open(
    temporary_configuration_path,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(configuration, output_file, indent=2)
temporary_configuration_path.replace(CONFIGURATION_PATH)

display(validation_summary)

print(f"Saved: {VALIDATION_SUMMARY_PATH}")
print(f"Saved: {CONFIGURATION_PATH}")
print(
    "\nTransition: Cell 32.12 will package and report "
    "the final Notebook 32 outputs."
)


In [ ]:
#@title Cell 32.12 - Package and report the final Notebook 32 outputs
# This cell records output checksums, creates one validated ZIP archive and
# reports the final three-model pathogen-out validation status.

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK32_DIRECTORY
    / "32_three_model_pathogen_out_validation_outputs.zip"
)
OUTPUT_MANIFEST_PATH = (
    NOTEBOOK32_RESULT_DIRECTORY
    / "32_output_manifest.json"
)

files_to_package = [
    ALIGNMENT_VALIDATION_PATH,
    MODEL3_PREDICTION_PATH,
    MODEL3_FOLD_CHECK_PATH,
    THREE_MODEL_PREDICTION_PATH,
    OVERALL_PERFORMANCE_PATH,
    OUTER_FOLD_PERFORMANCE_PATH,
    OUTER_FOLD_VARIATION_PATH,
    ANTIBIOTIC_PERFORMANCE_PATH,
    PAIRWISE_COMPARISON_PATH,
    BOOTSTRAP_PERFORMANCE_PATH,
    BOOTSTRAP_INTERVAL_PATH,
    PAIRWISE_BOOTSTRAP_INTERVAL_PATH,
    VALIDATION_FIGURE_PATH,
    VALIDATION_SUMMARY_PATH,
    CONFIGURATION_PATH,
]

missing_output_files = [
    path
    for path in files_to_package
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        f"Notebook 32 output files are missing: {missing_output_files}"
    )

overall_manifest_metrics = {
    row.model: {
        "mae": float(row.mae),
        "rmse": float(row.rmse),
        "pearson_r": float(row.pearson_r),
    }
    for row in overall_performance.itertuples(index=False)
}

output_manifest = {
    "notebook": 32,
    "models_compared": model_display_order,
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "observed_interactions": EXPECTED_INTERACTIONS,
    "antibiotics": EXPECTED_ANTIBIOTICS,
    "outer_folds": EXPECTED_OUTER_FOLDS,
    "overall_pathogen_out_performance": overall_manifest_metrics,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "files": [
        {
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in files_to_package
    ],
    "validation_status": "passed",
}

temporary_manifest_path = Path(
    str(OUTPUT_MANIFEST_PATH) + ".partial"
)
with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(output_manifest, output_file, indent=2)
temporary_manifest_path.replace(OUTPUT_MANIFEST_PATH)

files_to_package.append(OUTPUT_MANIFEST_PATH)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)
local_archive_path.unlink(missing_ok=True)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=3,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(local_archive_path, "r") as archive:
    damaged_member = archive.testzip()
    if damaged_member is not None:
        raise ValueError(
            f"The local Notebook 32 archive contains a damaged file: "
            f"{damaged_member}"
        )
    archived_members = set(archive.namelist())

expected_members = {
    path.name
    for path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The Notebook 32 archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(local_archive_path)
partial_archive_path = Path(
    str(FINAL_OUTPUT_ARCHIVE_PATH) + ".partial"
)
partial_archive_path.unlink(missing_ok=True)
shutil.copy2(local_archive_path, partial_archive_path)

if file_sha256(partial_archive_path) != local_archive_sha256:
    raise IOError(
        "The copied Notebook 32 archive does not match the local archive."
    )

partial_archive_path.replace(FINAL_OUTPUT_ARCHIVE_PATH)

with zipfile.ZipFile(FINAL_OUTPUT_ARCHIVE_PATH, "r") as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved Notebook 32 archive failed validation."
        )

final_summary = overall_performance[
    ["model", "mae", "rmse", "pearson_r"]
].copy()

display(final_summary.round(4))

display(pd.DataFrame([
    {"metric": "Common pathogens", "value": EXPECTED_MODEL_C_PATHOGENS},
    {"metric": "Common observed MIC values", "value": EXPECTED_INTERACTIONS},
    {"metric": "Outer pathogen folds", "value": EXPECTED_OUTER_FOLDS},
    {"metric": "Models compared", "value": EXPECTED_COMPARISON_MODELS},
    {"metric": "Files packaged", "value": len(files_to_package)},
    {"metric": "Final archive size (MB)", "value": round(FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size / (1024 ** 2), 3)},
    {"metric": "Notebook 32 validation status", "value": "passed"},
]))

print(f"Saved: {OUTPUT_MANIFEST_PATH}")
print(f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}")
print(
    "\nNotebook 32 completed successfully. Models 3, 3B and C "
    "were compared on the same five pathogen-out validation folds."
)
